# ESA GeoFM Challenge — TerraMind S2 Baseline
**Run all cells top to bottom: Runtime → Run all**

## Pipeline
1. Mount Drive & install deps
2. Discover patches
3. Compute/load normalisation stats
4. Dataset + DataLoader
5. SimpleFusionHead model
6. Loss & metrics
7. Training loop
8. Download test data
9. Predict & package submission


## 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q rasterio eotdl

In [ ]:
import os, re, json, random, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import rasterio
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
REPO      = Path('/content/drive/MyDrive/ESA_Challenge')
TRAIN     = REPO / 'train'
LABEL_DIR = TRAIN / 'labels'
TEST_DIR  = REPO / 'test'

# Single model — fastest to load (16x16 files, ~200KB each)
EMB_CONFIGS = {
    'terramind_s2_emb': {'n_ch': 768, 'res': 16},
}
TOTAL_CH = 768

# Test folder name mapping (train name -> test name)
TEST_EMB_MAP = {
    'terramind_s2_emb': 'terramind_test_s2_emb',
}

CFG = {
    'batch_size':      4,
    'num_workers':     0,
    'lr':              1e-4,
    'weight_decay':    1e-4,
    'epochs':          50,
    'val_split':       0.15,
    'seg_weight':      1.0,
    'height_weight':   1.0,
    'log1p_height':    True,
    'norm_stats_file': REPO / 'norm_stats.npy',
    'ckpt_path':       REPO / 'best_model.pth',
}
print(f'Total input channels: {TOTAL_CH}')

## 1 — Patch discovery

In [ ]:
def extract_num_id(stem):
    """Extract 4-digit numeric patch ID from any filename stem."""
    match = re.search(r'_(\d{4})_', stem)
    return match.group(1) if match else None

def get_num_ids(folder):
    """Return {num_id -> Path} for all .tif files in folder."""
    result = {}
    for f in Path(folder).iterdir():
        if f.is_file() and f.suffix == '.tif':
            nid = extract_num_id(f.stem)
            if nid:
                result[nid] = f
    return result

label_index = get_num_ids(LABEL_DIR)
emb_indexes = {name: get_num_ids(TRAIN / name) for name in EMB_CONFIGS}

for name, idx in emb_indexes.items():
    print(f'{name:25s}: {len(idx)} files')
print(f'{"labels":25s}: {len(label_index)} files')

# Intersect IDs across all folders
all_ids = set(label_index.keys())
for idx in emb_indexes.values():
    all_ids &= set(idx.keys())
all_ids = sorted(all_ids)

random.shuffle(all_ids)
n_val   = int(len(all_ids) * CFG['val_split'])
val_ids = all_ids[:n_val]
trn_ids = all_ids[n_val:]
print(f'\nTotal:{len(all_ids)}  Train:{len(trn_ids)}  Val:{len(val_ids)}')

## 2 — Normalisation stats

In [ ]:
def load_tif(path):
    try:
        with rasterio.open(path) as src:
            return src.read().astype(np.float32)
    except Exception:
        print(f'  CORRUPTED: {Path(path).name}')
        return None


def compute_norm_stats(patch_ids, emb_indexes, n_sample=20, save_path=None):
    sample = random.sample(patch_ids, min(n_sample, len(patch_ids)))
    stats  = {}
    for emb_name in tqdm(EMB_CONFIGS, desc='Norm stats'):
        accum_sum = None
        accum_sq  = None
        n_pixels  = 0
        skipped   = 0
        for num_id in sample:
            path = emb_indexes[emb_name].get(num_id)
            if path is None: skipped += 1; continue
            arr = load_tif(path)
            if arr is None: skipped += 1; continue
            c    = arr.shape[0]
            flat = arr.reshape(c, -1)
            valid = ~np.isnan(flat[0])
            if valid.sum() == 0: skipped += 1; continue
            if accum_sum is None:
                accum_sum = np.zeros(c, dtype=np.float64)
                accum_sq  = np.zeros(c, dtype=np.float64)
            accum_sum += np.nansum(flat, axis=1)
            accum_sq  += np.nansum(flat**2, axis=1)
            n_pixels  += valid.sum()
        if accum_sum is None or n_pixels == 0:
            n_ch = EMB_CONFIGS[emb_name]['n_ch']
            stats[emb_name] = {'mean': np.zeros(n_ch, dtype=np.float32),
                               'std':  np.ones(n_ch,  dtype=np.float32)}
            print(f'  {emb_name}: no valid files, using identity')
        else:
            mean = (accum_sum / n_pixels).astype(np.float32)
            std  = np.sqrt(np.maximum(
                accum_sq/n_pixels - mean.astype(np.float64)**2, 1e-6
            )).astype(np.float32)
            stats[emb_name] = {'mean': mean, 'std': std}
            print(f'  {emb_name:25s} mean[{mean.min():.3f},{mean.max():.3f}]  '
                  f'std[{std.min():.3f},{std.max():.3f}]  skipped:{skipped}')
    if save_path:
        np.save(save_path, stats)
        print(f'Saved to {save_path}')
    return stats


if CFG['norm_stats_file'].exists():
    norm_stats = np.load(CFG['norm_stats_file'], allow_pickle=True).item()
    # Subset to current model only
    norm_stats = {k: v for k, v in norm_stats.items() if k in EMB_CONFIGS}
    print('Loaded cached norm stats:')
    for name, s in norm_stats.items():
        print(f'  {name:25s} mean[{s["mean"].min():.3f},{s["mean"].max():.3f}]  '
              f'std[{s["std"].min():.3f},{s["std"].max():.3f}]')
    if not norm_stats:
        print('No matching stats found — recomputing...')
        norm_stats = compute_norm_stats(trn_ids, emb_indexes, n_sample=20,
                                        save_path=CFG['norm_stats_file'])
else:
    norm_stats = compute_norm_stats(trn_ids, emb_indexes, n_sample=20,
                                    save_path=CFG['norm_stats_file'])

## 3 — Dataset

In [ ]:
class GeoFMDataset(Dataset):

    def __init__(self, patch_ids, norm_stats, label_index, emb_indexes, augment=False):
        self.patch_ids   = patch_ids
        self.norm_stats  = norm_stats
        self.label_index = label_index
        self.emb_indexes = emb_indexes
        self.augment     = augment

    def _load_and_norm(self, path, emb_name):
        arr = load_tif(path)
        if arr is None:
            return np.zeros((EMB_CONFIGS[emb_name]['n_ch'], 16, 16), dtype=np.float32)
        mean = self.norm_stats[emb_name]['mean'][:, None, None]
        std  = self.norm_stats[emb_name]['std'][:, None, None]
        return np.where(np.isnan(arr), 0.0, (arr - mean) / std).astype(np.float32)

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, idx):
        num_id  = self.patch_ids[idx]
        tensors = []
        for emb_name in EMB_CONFIGS:
            arr = self._load_and_norm(self.emb_indexes[emb_name][num_id], emb_name)
            tensors.append(torch.from_numpy(arr))   # (768, 16, 16) — no upsample here
        fused = torch.cat(tensors, dim=0)

        # Labels
        lbl = torch.from_numpy(load_tif(self.label_index[num_id]))
        if lbl.shape[-2] != 256 or lbl.shape[-1] != 256:
            lbl = F.interpolate(lbl.unsqueeze(0), size=(256, 256),
                                mode='bilinear', align_corners=False).squeeze(0)
        seg_target    = lbl[:3].clamp(0, 1)
        height_target = lbl[3:4]
        if CFG['log1p_height']:
            height_target = torch.log1p(height_target.clamp(min=0))

        if self.augment:
            if random.random() > 0.5:
                fused         = torch.flip(fused,         dims=[-1])
                seg_target    = torch.flip(seg_target,    dims=[-1])
                height_target = torch.flip(height_target, dims=[-1])
            if random.random() > 0.5:
                fused         = torch.flip(fused,         dims=[-2])
                seg_target    = torch.flip(seg_target,    dims=[-2])
                height_target = torch.flip(height_target, dims=[-2])
            k = random.randint(0, 3)
            if k > 0:
                fused         = torch.rot90(fused,         k, dims=[-2, -1])
                seg_target    = torch.rot90(seg_target,    k, dims=[-2, -1])
                height_target = torch.rot90(height_target, k, dims=[-2, -1])

        return fused, seg_target, height_target


# Smoke test
ds_test        = GeoFMDataset(trn_ids[:4], norm_stats, label_index, emb_indexes)
fused, seg, ht = ds_test[0]
print(f'fused:  {fused.shape}  nan:{fused.isnan().any()}')
print(f'seg:    {seg.shape}    [{seg.min():.3f}, {seg.max():.3f}]')
print(f'height: {ht.shape}     [{ht.min():.3f}, {ht.max():.3f}]')

## 4 — Model

Simple MLP head — no UNet needed since input is already 16×16 abstract features.
Upsample happens *after* channel reduction to save memory.

In [ ]:
class SimpleFusionHead(nn.Module):
    """
    Input:  (B, 768, 16, 16)
    Output: seg    (B, 3, 256, 256)  — building/veg/water probability
            height (B, 1, 256, 256)  — log1p nDSM
    """
    def __init__(self, in_ch=768):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(in_ch, 256, 1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256,   128, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128,    64, 1, bias=False), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
        )
        self.seg_head    = nn.Conv2d(64, 3, 1)
        self.height_head = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        x      = self.head(x)                                          # (B, 64, 16, 16)
        x      = F.interpolate(x, size=(256, 256),
                               mode='bilinear', align_corners=False)   # (B, 64, 256, 256)
        seg    = torch.sigmoid(self.seg_head(x))                       # (B, 3, 256, 256)
        height = F.softplus(self.height_head(x))                       # (B, 1, 256, 256)
        return seg, height


model    = SimpleFusionHead(TOTAL_CH).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')

with torch.no_grad():
    dummy       = torch.randn(1, TOTAL_CH, 16, 16).to(DEVICE)
    seg_o, ht_o = model(dummy)
print(f'Output — seg:{seg_o.shape}  height:{ht_o.shape}')
if DEVICE == 'cuda':
    print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 5 — Loss & metrics

In [ ]:
class SegLoss(nn.Module):
    """Weighted MSE — upweights pixels where class is present."""
    def __init__(self, pw=(5.0, 2.0, 8.0)):
        super().__init__()
        self.pw = torch.tensor(pw).float()
    def forward(self, pred, tgt):
        pw = self.pw.to(pred.device).view(1, 3, 1, 1)
        w  = 1.0 + (tgt > 0.01).float() * pw
        return (w * (pred - tgt)**2).mean()

class HeightLoss(nn.Module):
    """Masked Huber loss — upweights above-ground pixels."""
    def __init__(self, delta=0.5, nzw=3.0):
        super().__init__()
        self.delta = delta; self.nzw = nzw
    def forward(self, pred, tgt):
        w = 1.0 + (tgt > 0).float() * (self.nzw - 1)
        return (w * F.huber_loss(pred, tgt, delta=self.delta, reduction='none')).mean()

class MultiTaskLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.seg_loss    = SegLoss()
        self.height_loss = HeightLoss()
    def forward(self, ps, ph, ts, th):
        ls = self.seg_loss(ps, ts)
        lh = self.height_loss(ph, th)
        return CFG['seg_weight']*ls + CFG['height_weight']*lh, ls.item(), lh.item()

criterion = MultiTaskLoss()

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, iou_sum, rmse_sum, n = 0, 0, 0, 0
    for fused, seg_tgt, ht_tgt in loader:
        fused   = fused.to(DEVICE)
        seg_tgt = seg_tgt.to(DEVICE)
        ht_tgt  = ht_tgt.to(DEVICE)
        ps, ph  = model(fused)
        loss, _, _ = criterion(ps, ph, seg_tgt, ht_tgt)
        total += loss.item()
        # IoU (buildings + veg + water)
        p = (ps > 0.3).float(); t = (seg_tgt > 0.3).float()
        inter = (p*t).sum(); union = (p+t-p*t).sum().clamp(min=1)
        iou_sum += (inter/union).item()
        # Height RMSE in metres
        ht_p = torch.expm1(ph); ht_t = torch.expm1(ht_tgt)
        mask = ht_t > 0
        if mask.sum() > 0:
            rmse_sum += ((ht_p[mask]-ht_t[mask])**2).mean().sqrt().item()
        n += 1
    return {'loss': total/n, 'iou': iou_sum/n, 'rmse': rmse_sum/n}

print('Loss and metrics ready.')

## 6 — Training

In [ ]:
trn_ds = GeoFMDataset(trn_ids, norm_stats, label_index, emb_indexes, augment=True)
val_ds = GeoFMDataset(val_ids, norm_stats, label_index, emb_indexes, augment=False)

trn_loader = DataLoader(trn_ds, batch_size=CFG['batch_size'], shuffle=True,
                        num_workers=0, pin_memory=False)
val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                        num_workers=0, pin_memory=False)

print(f'Train batches:{len(trn_loader)}  Val batches:{len(val_loader)}')

model     = SimpleFusionHead(TOTAL_CH).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = OneCycleLR(optimizer, max_lr=CFG['lr'],
                       steps_per_epoch=len(trn_loader),
                       epochs=CFG['epochs'], pct_start=0.1)

history  = {'trn_loss':[], 'val_loss':[], 'val_iou':[], 'val_rmse':[]}
best_val = float('inf')

# Resume from checkpoint if exists
start_epoch = 0
if CFG['ckpt_path'].exists():
    ckpt = torch.load(CFG['ckpt_path'], map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    best_val    = ckpt['val_loss']
    start_epoch = ckpt['epoch'] + 1
    print(f'Resumed from epoch {start_epoch} (val_loss={best_val:.4f})')

for epoch in range(start_epoch, CFG['epochs']):
    model.train()
    trn_sum = 0
    pbar = tqdm(trn_loader, desc=f'Ep {epoch+1}/{CFG["epochs"]}', leave=False)
    for fused, seg_tgt, ht_tgt in pbar:
        fused   = fused.to(DEVICE)
        seg_tgt = seg_tgt.to(DEVICE)
        ht_tgt  = ht_tgt.to(DEVICE)
        optimizer.zero_grad()
        ps, ph = model(fused)
        loss, ls, lh = criterion(ps, ph, seg_tgt, ht_tgt)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        trn_sum += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}', seg=f'{ls:.4f}', ht=f'{lh:.4f}')

    val_m    = evaluate(model, val_loader)
    trn_loss = trn_sum / len(trn_loader)
    history['trn_loss'].append(trn_loss)
    history['val_loss'].append(val_m['loss'])
    history['val_iou'].append(val_m['iou'])
    history['val_rmse'].append(val_m['rmse'])

    print(f'Ep {epoch+1:3d} | trn {trn_loss:.4f} | val {val_m["loss"]:.4f} | '
          f'IoU {val_m["iou"]:.4f} | RMSE {val_m["rmse"]:.2f}m')

    if val_m['loss'] < best_val:
        best_val = val_m['loss']
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'val_loss': best_val}, CFG['ckpt_path'])
        print(f'  --> Best saved (val={best_val:.4f})')

print('Training complete!')

## 7 — Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['trn_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history['val_iou'], color='green')
axes[1].set_title('Val mIoU')
axes[2].plot(history['val_rmse'], color='purple')
axes[2].set_title('Val Height RMSE (m)')
for ax in axes:
    ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8 — Download test data

Run once. Files saved to Drive permanently.

In [ ]:
import shutil

TEST_DRIVE = TEST_DIR / 'terramind_test_s2_emb'
TEST_DRIVE.mkdir(parents=True, exist_ok=True)

n_existing = len(list(TEST_DRIVE.glob('*.tif')))
print(f'Already on Drive: {n_existing} files')

if n_existing < 946:
    # Download catalog
    !eotdl datasets get embed2heights -v 1 -f

    cat = pd.read_parquet('/root/.cache/eotdl/datasets/embed2heights/catalog.v1.parquet')
    creds   = json.loads(open('/root/.cache/eotdl/creds.json').read())
    headers = {'Authorization': f'Bearer {creds["id_token"]}'}

    test_s2 = cat[cat['id'].str.contains('terramind_test_s2', na=False)]
    print(f'Files to download: {len(test_s2)}')

    failed = []
    for _, row in tqdm(test_s2.iterrows(), total=len(test_s2)):
        fname = Path(row['id']).name
        out   = TEST_DRIVE / fname
        if out.exists() and out.stat().st_size > 1000:
            continue
        try:
            url  = row['assets']['asset']['href']
            # Get presigned URL
            r    = requests.get(url, headers=headers, timeout=30)
            data = r.json()
            presigned = data['presigned_url']
            # Download actual file
            r2   = requests.get(presigned, timeout=60)
            r2.raise_for_status()
            out.write_bytes(r2.content)
        except Exception as e:
            failed.append((fname, str(e)))

    n_final = len(list(TEST_DRIVE.glob('*.tif')))
    print(f'Downloaded: {n_final} / 946  Failed: {len(failed)}')
    if failed:
        print('First failure:', failed[0])
else:
    print('All 946 test files already on Drive!')

## 9 — Predict & package submission

In [ ]:
# Index test files
test_emb_indexes = {'terramind_s2_emb': get_num_ids(TEST_DRIVE)}
test_ids         = sorted(test_emb_indexes['terramind_s2_emb'].keys())
print(f'Test patches: {len(test_ids)}')
assert len(test_ids) == 946, f'Expected 946, got {len(test_ids)}'


class TestDataset(Dataset):
    def __init__(self, patch_ids, norm_stats, emb_indexes):
        self.patch_ids   = patch_ids
        self.norm_stats  = norm_stats
        self.emb_indexes = emb_indexes

    def _load_and_norm(self, path, emb_name):
        arr = load_tif(path)
        if arr is None:
            return np.zeros((EMB_CONFIGS[emb_name]['n_ch'], 16, 16), dtype=np.float32)
        mean = self.norm_stats[emb_name]['mean'][:, None, None]
        std  = self.norm_stats[emb_name]['std'][:, None, None]
        return np.where(np.isnan(arr), 0.0, (arr-mean)/std).astype(np.float32)

    def __len__(self): return len(self.patch_ids)

    def __getitem__(self, idx):
        num_id  = self.patch_ids[idx]
        tensors = []
        for emb_name in EMB_CONFIGS:
            arr = self._load_and_norm(self.emb_indexes[emb_name][num_id], emb_name)
            tensors.append(torch.from_numpy(arr))
        return torch.cat(tensors, dim=0), num_id


# Load best checkpoint
ckpt = torch.load(CFG['ckpt_path'], map_location=DEVICE)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded checkpoint epoch {ckpt["epoch"]+1}  val_loss={ckpt["val_loss"]:.4f}')

# Run predictions
SUBMIT_TIF = REPO / 'submission_tif'
SUBMIT_TIF.mkdir(exist_ok=True)

test_ds     = TestDataset(test_ids, norm_stats, test_emb_indexes)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=0)

with torch.no_grad():
    for fused, num_ids in tqdm(test_loader, desc='Predicting'):
        fused     = fused.to(DEVICE)
        pred_seg, pred_ht = model(fused)
        pred_ht   = torch.expm1(pred_ht.clamp(min=0))   # back to metres
        out       = torch.cat([pred_seg, pred_ht], dim=1).cpu().numpy()
        for b, num_id in enumerate(num_ids):
            arr = out[b].astype(np.float32)
            with rasterio.open(SUBMIT_TIF/f'pred_{num_id}.tif', 'w',
                               driver='GTiff', height=256, width=256,
                               count=4, dtype='float32') as dst:
                dst.write(arr)

print(f'Saved {len(list(SUBMIT_TIF.glob("*.tif")))} tif predictions')

# Convert to .npy with correct naming and zip
# Format: submission.zip/predictions/3335_MM_2022.npy
SUBMIT_NPY = REPO / 'submission_npy'
SUBMIT_NPY.mkdir(exist_ok=True)

for num_id, path in tqdm(test_emb_indexes['terramind_s2_emb'].items(), desc='Converting'):
    # s2_3335_MM_2022_embeddings -> 3335_MM_2022
    parts = path.stem.split('_')
    name  = '_'.join(parts[1:-1]) + '.npy'
    pred  = load_tif(SUBMIT_TIF / f'pred_{num_id}.tif')
    if pred is not None:
        np.save(SUBMIT_NPY / name, pred)

n_npy = len(list(SUBMIT_NPY.glob('*.npy')))
print(f'Converted: {n_npy} npy files')

# Zip
ZIP_PATH = REPO / 'submission.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for npy in tqdm(sorted(SUBMIT_NPY.glob('*.npy')), desc='Zipping'):
        zf.write(npy, f'predictions/{npy.name}')

print(f'\nsubmission.zip ready: {ZIP_PATH.stat().st_size/1e6:.1f} MB')
print(f'Files inside: {n_npy} / 946')

# Sanity check
sample = np.load(sorted(SUBMIT_NPY.glob('*.npy'))[0])
print(f'\nSample shape: {sample.shape}')
print(f'Building  [{sample[0].min():.3f}, {sample[0].max():.3f}]')
print(f'Veg       [{sample[1].min():.3f}, {sample[1].max():.3f}]')
print(f'Water     [{sample[2].min():.3f}, {sample[2].max():.3f}]')
print(f'Height(m) [{sample[3].min():.3f}, {sample[3].max():.3f}]')